# 01: Exploratory Data Analysis (EDA)

This notebook is the **first step** in the Student Performance Prediction project.

**Goals of this notebook:**
1. Load the raw dataset
2. Inspect its structure and quality
3. Clean and validate the data (handle missing values, duplicates)
4. Explore relationships between features and the target variable (`G3`)
5. Visualize key patterns through professional plots
6. Save cleaned data for use in the next notebook (`02_preprocessing.ipynb`)

---

**Dataset:** [UCI Student Performance Dataset](https://archive.ics.uci.edu/ml/datasets/student+performance)
**Target variable:** `G3` — final grade (0–20 scale)


## 1. Setup & Imports

We start by importing the libraries needed for data handling and visualization, and configuring our plotting style for consistent, professional-looking charts.

In [ ]:
# Core libraries
import pandas as pd
import numpy as np

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Allow imports from the src/ folder (project root must be one level up)
import sys
sys.path.append("..")

from src.data_loader import load_data, get_basic_info
from src.preprocessing import add_pass_fail
from src.visualize import plot_grade_distribution, plot_correlation_heatmap

# Configure plot style for consistent, professional visuals
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.figsize"] = (10, 6)

%matplotlib inline


## 2. Load the Dataset

The dataset is stored in `data/raw/student-mat.csv`. The UCI dataset uses a **semicolon (`;`)** as the delimiter, which our `load_data()` helper function handles automatically.

In [ ]:
# Load the dataset
df = load_data("../data/raw/student-mat.csv")

# Preview the first few rows
df.head()


## 3. Initial Data Inspection

Before doing anything else, we need to understand:
- How many rows and columns we have
- What data types each column has
- Whether there are missing values or duplicate rows

This step ensures we know exactly what we're working with before any cleaning or analysis.

In [ ]:
# Display structured overview of the dataset
get_basic_info(df)


In [ ]:
# Statistical summary of numerical columns
df.describe()


In [ ]:
# List all column names grouped by data type
print("Numerical columns:")
print(df.select_dtypes(include=[np.number]).columns.tolist())

print("\nCategorical columns:")
print(df.select_dtypes(include=["object"]).columns.tolist())


## 4. Data Cleaning & Handling Missing Values

**Findings:**
- The UCI Student Performance dataset is generally clean and well-curated, with **no missing values** in most distributions.
- However, we still run explicit checks as a best practice — real-world pipelines should **never assume** data is clean.

**Our cleaning strategy:**
1. Check for missing values column-by-column
2. Check for and remove exact duplicate rows
3. Validate that grade columns (`G1`, `G2`, `G3`) fall within the expected 0–20 range
4. Confirm categorical columns contain only expected category values

In [ ]:
# Step 1: Check for missing values per column
missing = df.isnull().sum()
missing = missing[missing > 0]

if missing.empty:
    print("✅ No missing values found in the dataset.")
else:
    print("⚠️ Missing values detected:")
    print(missing)


In [ ]:
# Step 2: Check for and remove duplicate rows
duplicate_count = df.duplicated().sum()
print(f"Duplicate rows found: {duplicate_count}")

if duplicate_count > 0:
    df = df.drop_duplicates()
    print(f"✅ Duplicates removed. New shape: {df.shape}")
else:
    print("✅ No duplicate rows to remove.")


In [ ]:
# Step 3: Validate that grade columns are within the expected 0-20 range
grade_cols = ["G1", "G2", "G3"]

for col in grade_cols:
    out_of_range = df[(df[col] < 0) | (df[col] > 20)]
    print(f"{col}: min={df[col].min()}, max={df[col].max()}, out-of-range rows={len(out_of_range)}")


In [ ]:
# Step 4: Spot-check categorical columns for unexpected values
categorical_cols = df.select_dtypes(include=["object"]).columns

for col in categorical_cols:
    unique_vals = df[col].unique()
    print(f"{col:<12}: {unique_vals}")


**Conclusion:** The dataset contains **no missing values** and **no duplicate rows**, and all grade values fall within the expected 0–20 range. Categorical columns contain only valid, expected categories. The data is clean and ready for analysis.

## 5. Create the Pass/Fail Target (for Classification)

In addition to predicting the exact final grade (`G3`) as a **regression** problem, we'll also frame this as a **classification** problem:

> **Pass:** `G3 >= 10`
> **Fail:** `G3 < 10`

This dual framing gives us flexibility to demonstrate multiple types of ML models later in the project.

In [ ]:
# Add the binary pass_fail column
df = add_pass_fail(df, grade_col="G3", threshold=10)

# Preview the new column
df[["G1", "G2", "G3", "pass_fail"]].head()


## 6. Target Variable Analysis

Let's explore the distribution of our target variable `G3` (final grade) and the derived `pass_fail` classification label.

**What we're looking for:**
- Is the grade distribution roughly normal, skewed, or multimodal?
- Is there a class imbalance between Pass and Fail students?

In [ ]:
# Plot the distribution of final grades and pass/fail split
plot_grade_distribution(df, grade_col="G3")


In [ ]:
# Print exact value counts for pass/fail
print(df["pass_fail"].value_counts())
print("\nPass rate: {:.1f}%".format(df["pass_fail"].mean() * 100))


**Observation:** The grade distribution is roughly bell-shaped with a notable cluster of students scoring `G3 = 0` (likely students who did not take the final exam or withdrew). The Pass/Fail split shows a reasonable balance, though Fail (G3 < 10) is the minority class — something to keep in mind during model evaluation.

## 7. Numerical Feature Distributions

Let's visualize the distributions of key numerical features:
- `age` — Student age
- `studytime` — Weekly study time (1=<2hrs, 2=2-5hrs, 3=5-10hrs, 4=>10hrs)
- `failures` — Number of past class failures
- `absences` — Number of school absences
- `G1`, `G2` — First and second period grades

In [ ]:
# Plot histograms for key numerical features
numerical_features = ["age", "studytime", "failures", "absences", "G1", "G2"]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Distributions of Key Numerical Features", fontsize=16, fontweight="bold")

for ax, col in zip(axes.flatten(), numerical_features):
    sns.histplot(df[col], bins=15, kde=True, ax=ax, color="#4C72B0")
    ax.set_title(col)
    ax.set_xlabel("")

plt.tight_layout()
plt.savefig("../reports/figures/numerical_distributions.png", bbox_inches="tight", dpi=150)
plt.show()


**Observations:**
- `absences` is heavily right-skewed — most students have few absences, but some have many
- `failures` shows most students have **zero** past failures
- `G1` and `G2` are roughly normally distributed and closely resemble each other (and `G3`), suggesting strong predictive power for the final grade

## 8. Correlation Analysis

Now we examine how numerical features correlate with each other and, most importantly, with our target variable `G3`.

A correlation heatmap helps us:
- Identify features strongly related to the final grade
- Spot multicollinearity between features (which we may need to address during preprocessing)

In [ ]:
# Plot correlation heatmap of numerical features
plot_correlation_heatmap(df)


In [ ]:
# Print the features most correlated with G3 (final grade)
numeric_df = df.select_dtypes(include=[np.number])
correlations = numeric_df.corr()["G3"].sort_values(ascending=False)

print("Top correlations with G3 (Final Grade):\n")
print(correlations)


**Key Insight:** As expected, `G1` and `G2` (first and second period grades) show the **strongest correlation** with `G3`. This makes sense — past performance is a strong predictor of future performance. `failures` shows a notable **negative correlation**, while `studytime` and parental education levels (`Medu`, `Fedu`) show weaker positive correlations.

## 9. Categorical Feature Analysis

Let's explore how categorical features relate to student performance. We'll use boxplots to compare the distribution of `G3` across different categories.

**Features explored:**
- `sex` — Student gender
- `school` — School attended
- `internet` — Internet access at home
- `romantic` — In a romantic relationship
- `higher` — Wants to pursue higher education
- `schoolsup` — Extra educational school support

In [ ]:
# Boxplots: Categorical features vs Final Grade (G3)
categorical_features = ["sex", "school", "internet", "romantic", "higher", "schoolsup"]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Final Grade (G3) by Categorical Features", fontsize=16, fontweight="bold")

for ax, col in zip(axes.flatten(), categorical_features):
    sns.boxplot(data=df, x=col, y="G3", ax=ax, palette="muted")
    ax.set_title(f"G3 by {col}")
    ax.set_xlabel("")

plt.tight_layout()
plt.savefig("../reports/figures/categorical_vs_grade.png", bbox_inches="tight", dpi=150)
plt.show()


**Observations:**
- Students who want to pursue **higher education** (`higher = yes`) show notably higher final grades — a strong signal of motivation
- Students receiving **extra school support** (`schoolsup = yes`) tend to have slightly lower grades, likely because this support is targeted at students who are already struggling
- Differences across `sex` and `school` are relatively modest
- Students in romantic relationships show a slight tendency toward lower grades

## 10. Study Time & Failures vs. Final Grade

These two features are commonly cited as strong academic performance indicators. Let's verify this visually.

In [ ]:
# Study time vs Final Grade
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Study Habits & Academic History vs Final Grade", fontsize=16, fontweight="bold")

sns.boxplot(data=df, x="studytime", y="G3", ax=axes[0], palette="Blues")
axes[0].set_title("Final Grade by Weekly Study Time")
axes[0].set_xlabel("Study Time (1=<2hrs, 4=>10hrs)")

sns.boxplot(data=df, x="failures", y="G3", ax=axes[1], palette="Reds")
axes[1].set_title("Final Grade by Past Class Failures")
axes[1].set_xlabel("Number of Past Failures")

plt.tight_layout()
plt.savefig("../reports/figures/studytime_failures_vs_grade.png", bbox_inches="tight", dpi=150)
plt.show()


**Observations:**
- There's a **clear positive trend**: more study time generally corresponds to higher final grades
- Past **failures show a strong negative relationship** with final grades — students with 1+ past failures score noticeably lower on average than those with zero failures

## 11. Lifestyle Factors: Alcohol Consumption vs Grades

The dataset includes weekday (`Dalc`) and weekend (`Walc`) alcohol consumption ratings (1=very low to 5=very high). Let's see how these relate to academic performance.

In [ ]:
# Alcohol consumption vs Final Grade
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Alcohol Consumption vs Final Grade", fontsize=16, fontweight="bold")

sns.boxplot(data=df, x="Dalc", y="G3", ax=axes[0], palette="YlOrRd")
axes[0].set_title("Final Grade by Workday Alcohol Consumption")
axes[0].set_xlabel("Workday Alcohol Consumption (1=Low, 5=High)")

sns.boxplot(data=df, x="Walc", y="G3", ax=axes[1], palette="YlOrRd")
axes[1].set_title("Final Grade by Weekend Alcohol Consumption")
axes[1].set_xlabel("Weekend Alcohol Consumption (1=Low, 5=High)")

plt.tight_layout()
plt.savefig("../reports/figures/alcohol_vs_grade.png", bbox_inches="tight", dpi=150)
plt.show()


**Observation:** There is a mild downward trend in final grades as alcohol consumption increases, particularly for workday consumption (`Dalc`). The effect is more pronounced at the highest consumption levels, though sample sizes for those groups are small.

## 12. Save Cleaned Dataset

Finally, we save our cleaned dataset (with the new `pass_fail` column added) to `data/processed/` so it can be used directly in the next notebook: **`02_preprocessing.ipynb`**.

In [ ]:
# Save the cleaned dataset for the next stage of the pipeline
output_path = "../data/processed/student_clean.csv"
df.to_csv(output_path, index=False)

print(f"✅ Cleaned dataset saved to: {output_path}")
print(f"Final shape: {df.shape}")


## 13. Summary of Key Findings

| Finding | Insight |
|---|---|
| **No missing values / duplicates** | Dataset is clean and analysis-ready |
| **G1, G2 strongly predict G3** | Past grades are the strongest predictors of final performance |
| **Failures negatively impact grades** | Students with past failures score significantly lower |
| **Study time correlates positively** | More weekly study time → higher grades |
| **Higher education aspiration matters** | Students aiming for higher education outperform peers |
| **Alcohol consumption has mild negative effect** | Especially on workdays, at high consumption levels |
| **Pass rate** | The majority of students pass (G3 ≥ 10), with some class imbalance |

---

### ➡️ Next Step
Proceed to **`02_preprocessing.ipynb`** to encode categorical variables, scale features, and prepare train/test splits for modeling.